In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from scipy.sparse import hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

In [2]:
ROOT = Path.cwd().parent

DATASETS = ROOT / "datasets"
MISCONCEPTIONS = DATASETS / "eedi_misconceptions"

MODELS = ROOT / "models"
MODELS.mkdir(exist_ok=True)

print("Project root:", ROOT)
print("Dataset:", MISCONCEPTIONS)
print("Models:", MODELS)

Project root: c:\Users\Akshat\Desktop\Over_Here\Projects\Quintet
Dataset: c:\Users\Akshat\Desktop\Over_Here\Projects\Quintet\datasets\eedi_misconceptions
Models: c:\Users\Akshat\Desktop\Over_Here\Projects\Quintet\models


In [3]:
miscon = pd.read_csv(
    MISCONCEPTIONS / "train.csv"
)

print("Raw dataset shape:", miscon.shape)

print("\nColumns:")
print(miscon.columns.tolist())

display(miscon.head(3))

Raw dataset shape: (1869, 15)

Columns:
['QuestionId', 'ConstructId', 'ConstructName', 'SubjectId', 'SubjectName', 'CorrectAnswer', 'QuestionText', 'AnswerAText', 'AnswerBText', 'AnswerCText', 'AnswerDText', 'MisconceptionAId', 'MisconceptionBId', 'MisconceptionCId', 'MisconceptionDId']


,QuestionId,ConstructId,ConstructName,SubjectId,SubjectName,CorrectAnswer,QuestionText,AnswerAText,AnswerBText,AnswerCText,AnswerDText,MisconceptionAId,MisconceptionBId,MisconceptionCId,MisconceptionDId
0,0,856,Use the order of operations to carry out calcu...,33,BIDMAS,A,\[\n3 \times 2+4-5\n\]\nWhere do the brackets ...,\( 3 \times(2+4)-5 \),\( 3 \times 2+(4-5) \),\( 3 \times(2+4-5) \),Does not need brackets,NaN,NaN,NaN,1672.0
1,1,1612,Simplify an algebraic fraction by factorising ...,1077,Simplifying Algebraic Fractions,D,"Simplify the following, if possible: \( \frac{...",\( m+1 \),\( m+2 \),\( m-1 \),Does not simplify,2142.0,143.0,2142.0,NaN
2,2,2774,Calculate the range from a list of data,339,Range and Interquartile Range from a List of Data,B,Tom and Katie are discussing the \( 5 \) plant...,Only\nTom,Only\nKatie,Both Tom and Katie,Neither is correct,1287.0,NaN,1287.0,1073.0


In [4]:
rows = []

for _, row in miscon.iterrows():

    options = {
        "A": row["AnswerAText"],
        "B": row["AnswerBText"],
        "C": row["AnswerCText"],
        "D": row["AnswerDText"]
    }

    labels = {
        "A": row["MisconceptionAId"],
        "B": row["MisconceptionBId"],
        "C": row["MisconceptionCId"],
        "D": row["MisconceptionDId"]
    }

    for option in options:

        if pd.notna(labels[option]):

            rows.append({
                "QuestionId": row["QuestionId"],
                "QuestionText": row["QuestionText"],
                "SelectedAnswer": options[option],
                "CorrectAnswer": row["CorrectAnswer"],
                "SubjectName": row["SubjectName"],
                "ConstructName": row["ConstructName"],
                "MisconceptionId": int(labels[option])
            })

miscon_train = pd.DataFrame(rows)

print("Generated examples:", miscon_train.shape)

display(miscon_train.head(3))

Generated examples: (4370, 7)


,QuestionId,QuestionText,SelectedAnswer,CorrectAnswer,SubjectName,ConstructName,MisconceptionId
0,0,\[\n3 \times 2+4-5\n\]\nWhere do the brackets ...,Does not need brackets,A,BIDMAS,Use the order of operations to carry out calcu...,1672
1,1,"Simplify the following, if possible: \( \frac{...",\( m+1 \),D,Simplifying Algebraic Fractions,Simplify an algebraic fraction by factorising ...,2142
2,1,"Simplify the following, if possible: \( \frac{...",\( m+2 \),D,Simplifying Algebraic Fractions,Simplify an algebraic fraction by factorising ...,143


In [5]:
before = len(miscon_train)

miscon_train = miscon_train.drop_duplicates(
    subset=[
        "QuestionId",
        "SelectedAnswer",
        "MisconceptionId"
    ]
).reset_index(drop=True)

after = len(miscon_train)

print("Before:", before)
print("After:", after)
print("Duplicates removed:", before - after)

Before: 4370
After: 4365
Duplicates removed: 5


In [7]:
def clean_text(text):
    text = str(text)
    text = " ".join(text.split())
    return text.strip()


for column in [
    "QuestionText",
    "SelectedAnswer",
    "CorrectAnswer",
    "SubjectName",
    "ConstructName"
]:
    miscon_train[column] = miscon_train[column].apply(clean_text)


miscon_train["text"] = (
    "subject: " + miscon_train["SubjectName"] +
    " construct: " + miscon_train["ConstructName"] +
    " question: " + miscon_train["QuestionText"] +
    " correct answer: " + miscon_train["CorrectAnswer"] +
    " student answer: " + miscon_train["SelectedAnswer"]
)

display(
    miscon_train[
        ["QuestionId", "text", "MisconceptionId"]
    ].head(3)
)

,QuestionId,text,MisconceptionId
0,0,subject: BIDMAS construct: Use the order of op...,1672
1,1,subject: Simplifying Algebraic Fractions const...,2142
2,1,subject: Simplifying Algebraic Fractions const...,143


In [8]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        miscon_train,
        groups=miscon_train["QuestionId"]
    )
)

train_data = miscon_train.iloc[
    train_idx
].reset_index(drop=True)

val_data = miscon_train.iloc[
    val_idx
].reset_index(drop=True)

print("Training examples:", len(train_data))
print("Validation examples:", len(val_data))

train_questions = set(train_data["QuestionId"])
val_questions = set(val_data["QuestionId"])

print("Training questions:", len(train_questions))
print("Validation questions:", len(val_questions))
print("Question overlap:", len(train_questions & val_questions))

Training examples: 3498
Validation examples: 867
Training questions: 1495
Validation questions: 374
Question overlap: 0


In [9]:
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.98,
    sublinear_tf=True,
    max_features=30000
)

X_train_word = word_vectorizer.fit_transform(
    train_data["text"]
)

X_val_word = word_vectorizer.transform(
    val_data["text"]
)

print("Word TF-IDF:")
print("Training:", X_train_word.shape)
print("Validation:", X_val_word.shape)

Word TF-IDF:
Training: (3498, 19109)
Validation: (867, 19109)


In [10]:
char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=1,
    max_features=30000,
    sublinear_tf=True
)

X_train_char = char_vectorizer.fit_transform(
    train_data["text"]
)

X_val_char = char_vectorizer.transform(
    val_data["text"]
)

print("Character TF-IDF:")
print("Training:", X_train_char.shape)
print("Validation:", X_val_char.shape)

Character TF-IDF:
Training: (3498, 30000)
Validation: (867, 30000)


In [11]:
X_train = hstack([
    X_train_word,
    X_train_char
])

X_val = hstack([
    X_val_word,
    X_val_char
])

y_train = train_data["MisconceptionId"]
y_val = val_data["MisconceptionId"]

print("Combined training matrix:", X_train.shape)
print("Combined validation matrix:", X_val.shape)


final_model = LinearSVC(
    C=2.0,
    class_weight="balanced",
    random_state=42
)

final_model.fit(
    X_train,
    y_train
)

print("\nFinal misconception classifier trained.")


# Save model
joblib.dump(
    final_model,
    MODELS / "misconception_classifier.joblib"
)

# Save vectorizers
joblib.dump(
    word_vectorizer,
    MODELS / "misconception_word_tfidf.joblib"
)

joblib.dump(
    char_vectorizer,
    MODELS / "misconception_char_tfidf.joblib"
)

print("\nSaved final models:")
print(MODELS / "misconception_classifier.joblib")
print(MODELS / "misconception_word_tfidf.joblib")
print(MODELS / "misconception_char_tfidf.joblib")

Combined training matrix: (3498, 49109)
Combined validation matrix: (867, 49109)

Final misconception classifier trained.

Saved final models:
c:\Users\Akshat\Desktop\Over_Here\Projects\Quintet\models\misconception_classifier.joblib
c:\Users\Akshat\Desktop\Over_Here\Projects\Quintet\models\misconception_word_tfidf.joblib
c:\Users\Akshat\Desktop\Over_Here\Projects\Quintet\models\misconception_char_tfidf.joblib


In [12]:
y_pred = final_model.predict(X_val)

print("Predictions generated:", len(y_pred))

Predictions generated: 867


In [13]:
top1_accuracy = accuracy_score(y_val, y_pred)

print(f"Top-1 Accuracy: {top1_accuracy:.2%}")

Top-1 Accuracy: 21.11%


In [14]:
decision_scores = final_model.decision_function(X_val)

classes = final_model.classes_

top_k = 5

top_indices = np.argsort(
    decision_scores,
    axis=1
)[:, -top_k:][:, ::-1]

top_predictions = classes[top_indices]

print("Top-5 predictions generated.")
print("Shape:", top_predictions.shape)

Top-5 predictions generated.
Shape: (867, 5)


In [15]:
def top_k_accuracy(y_true, predictions, k):
    return np.mean([
        true_label in predictions[i, :k]
        for i, true_label in enumerate(y_true)
    ])


top3_accuracy = top_k_accuracy(
    y_val.to_numpy(),
    top_predictions,
    3
)

top5_accuracy = top_k_accuracy(
    y_val.to_numpy(),
    top_predictions,
    5
)

print(f"Top-1 Accuracy: {top1_accuracy:.2%}")
print(f"Top-3 Accuracy: {top3_accuracy:.2%}")
print(f"Top-5 Accuracy: {top5_accuracy:.2%}")

Top-1 Accuracy: 21.11%
Top-3 Accuracy: 38.41%
Top-5 Accuracy: 42.91%


In [16]:
train_labels = set(y_train.unique())
val_labels = set(y_val.unique())

unseen_labels = val_labels - train_labels
unseen_examples = (~y_val.isin(train_labels)).sum()

print("=" * 50)
print("FINAL MISCONCEPTION MODEL")
print("=" * 50)

print(f"Training examples: {len(y_train)}")
print(f"Validation examples: {len(y_val)}")
print(f"Training classes: {len(train_labels)}")
print(f"Validation classes: {len(val_labels)}")

print(f"\nUnseen validation classes: {len(unseen_labels)}")
print(f"Unseen validation examples: {unseen_examples}")

print("\nPERFORMANCE")
print(f"Top-1: {top1_accuracy:.2%}")
print(f"Top-3: {top3_accuracy:.2%}")
print(f"Top-5: {top5_accuracy:.2%}")

FINAL MISCONCEPTION MODEL
Training examples: 3498
Validation examples: 867
Training classes: 1414
Validation classes: 562

Unseen validation classes: 190
Unseen validation examples: 224

PERFORMANCE
Top-1: 21.11%
Top-3: 38.41%
Top-5: 42.91%
